In [ ]:
import ee
from tgbs_rs.config.config import GEE_PROJECT

In [5]:
# Authenticate and Initialize Earth Engine
ee.Authenticate()
ee.Initialize(project= "GEE_PROJECT")

In [ ]:
from tgbs_rs.config.config import (
    CURRENT_START, 
    CURRENT_END, 
    LAND_METRICS_DRIVE_FOLDER,
)
from tgbs_rs.data.sensors.dynamic_world.dynamic_world_preprocessing import build_annual_dw_woody_cover_collection
from tgbs_rs.utils import build_default_sites_featurecollection
from tgbs_rs.utils import buffer_sites_fc
from tgbs_rs.io import export_image_collection_to_drive

In [7]:
# Build Site Feature Collection
sites_fc = build_default_sites_featurecollection()

# Buffer each feature by 0.5km
buffered_fc = buffer_sites_fc(sites_fc, buffer_m=500)
features = buffered_fc.getInfo()["features"]

##### Build Binary Woody vs. Non-Woody Landcover for each site

In [8]:
sites_dict = {}

for feat in features:
    props = feat["properties"]
    site_id = props["site_id"]

    buffered_feature = ee.Feature(feat)
    buffered_aoi = buffered_feature.geometry()

    annual_col = build_annual_dw_woody_cover_collection(
        aoi=buffered_aoi,
        start_date=CURRENT_START,
        end_date=CURRENT_END,
    )

    sites_dict[site_id] = {
        "feature": buffered_feature,
        "aoi": buffered_aoi,
        "site_id": props.get("site_id"),
        "site_name": props.get("site_name"),
        "site_category": props.get("site_category"),
        "collection": annual_col,
    }

##### Check Site IDs, size, and site collection years

In [ ]:
site_ids = list(sites_dict.keys())

for site in site_ids:
    col = sites_dict[site]["collection"]

    print(site, col.size().getInfo())
    print(col.aggregate_array("year").getInfo())

##### Select Each Site and Export Annual Collection

In [ ]:
'''
site_info = sites_dict['ks_rehab']
site_info = sites_dict['buda']
site_info = sites_dict['gogoni']
site_info = sites_dict['shimba_hills']
site_info = sites_dict['degraded_1']
site_info = sites_dict['degraded_2']
site_info = sites_dict['degraded_3']
'''

In [ ]:
'''
tasks = export_image_collection_to_drive(
    collection=site_info["collection"],
    aoi=site_info["aoi"],
    folder=LAND_METRICS_DRIVE_FOLDER,
    file_prefix=site_info['site_id'],
    scale=10,
    band_names=["woody_cover"],
    description_prefix=f"{site_info['site_id']}_dw_woody_cover",
    date_property=None,
    fallback_property="year",
    file_suffix="dw_woody_cover",
)
'''